# 缺失值填充方法

本notebook专门介绍几种高级缺失值填充方法：

1. 条件平均值填充

2. 热卡填充（Hot deck imputation）

3. KNN填充

4. 回归填充（Regression）

5. 多重插补（Multiple Imputation）

## 1. 条件平均值填充

与其相似的另一种方法叫**条件平均值填充法**（Conditional Mean Completer）。在该方法中，用于求平均的值并不是从数据集的所有对象中取，而是从与该对象具有**相同决策属性值**的对象中取得。

In [ ]:
import pandas as pd
import numpy as np

# 构造示例数据
df = pd.DataFrame({
    "用户ID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十", "钱十一", "陈十二"],
    "性别": ["男", "女", "男", "女", "男", "女", "男", "女", "男", "女"],
    "年龄": [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", "Beijing", "hangzhou", "Nanjing", "shenzhen", "Beijing"],
    "年收入": [80000, 120000, 75000, 95000, 120000, 110000, 98000, 105000, 88000, 92000],
    "上次消费金额": [200, 500, 300, np.nan, 500, 400, 250, 350, 280, 320]
})

print("原始数据：")
print(df)

# 按【性别】分组，填充【年龄】缺失值
df["年龄"] = df.groupby("性别")["年龄"].transform(
    lambda x: x.fillna(x.mean())
)

# 按【性别+城市】双条件分组，填充年龄缺失值
df["年龄"] = df.groupby(["性别", "城市"])["年龄"].transform(
    lambda x: x.fillna(x.mean())
)

# 按【城市】分组，填充【上次消费金额】缺失值
df["上次消费金额"] = df.groupby("城市")["上次消费金额"].transform(
    lambda x: x.fillna(x.mean())
)

print("\n条件平均值填充后的数据：")
print(df)

## 2. 热卡填充（Hot deck imputation，或就近补齐）
-------------------------------------------------------------------

对于一个包含空值的对象，热卡填充法在完整数据中找到**一个**与它最相似的对象，然后用这个**相似对象的值来进行填充**。

不同的问题可能会选用不同的标准来对相似进行判定。该方法概念上很简单，且利用了数据间的关系来进行空值估计。

这个方法的缺点在于难以定义相似标准，**主观因素较多**。

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import LabelEncoder

# 构造示例数据
df = pd.DataFrame({
    "用户ID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十", "钱十一", "陈十二"],
    "性别": ["男", "女", "男", "女", "男", "女", "男", "女", "男", "女"],
    "年龄": [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", "Beijing", "hangzhou", "Nanjing", "shenzhen", "Beijing"],
    "年收入": [80000, 120000, 75000, 95000, 120000, 110000, 98000, 105000, 88000, 92000],
    "上次消费金额": [200, 500, 300, np.nan, 500, 400, 250, 350, 280, 320]
})

print("原始数据：")
print(df)


def hot_deck_impute(df, target_col):
    """
    热卡填充法
    参数：
        df: DataFrame
        target_col: 需要填充的列名
    返回：
        填充后的DataFrame
    """
    df_hot = df.copy()
    
    # 城市编码
    le = LabelEncoder()
    df_hot["城市"] = df_hot["城市"].fillna("未知")
    df_hot["城市"] = le.fit_transform(df_hot["城市"])
    
    # 性别编码
    df_hot["性别"] = le.fit_transform(df_hot["性别"])
    
    # 完整数据
    complete = df_hot.dropna()
    # 缺失数据
    missing = df_hot[df_hot[target_col].isnull()]

    if len(missing) == 0:
        return df_hot

    # 计算欧氏距离，找最近1个样本
    dists = pairwise_distances(
        missing.drop(target_col, axis=1),
        complete.drop(target_col, axis=1),
        metric="euclidean"
    )

    # 逐个填充
    for i, idx in enumerate(missing.index):
        nearest = np.argmin(dists[i])
        fill_val = complete.iloc[nearest][target_col]
        df_hot.loc[idx, target_col] = fill_val
    return df_hot


# 对年龄做热卡填充
df_hot = hot_deck_impute(df, "年龄")

# 对上次消费金额做热卡填充
df_hot = hot_deck_impute(df_hot, "上次消费金额")

print("\n热卡填充后的数据：")
print(df_hot)

## 3. KNN填充
--------------------------------------------------------------------

先根据欧式距离或相关分析来确定距离具有缺失数据样本最近的K个样本，将这K个值加权平均来估计该样本的缺失数据。

根据数据类型的不同，距离度量也不尽相同：
1、连续数据：最常用的距离度量有欧氏距离，曼哈顿距离以及余弦距离。
2、分类数据：汉明（Hamming）距离在这种情况比较常用。对于所有分类属性的取值，如果两个数据点的值不同，则距离加一。汉明距离实际上与属性间不同取值的数量一致。

KNN算法的一个明显缺点是，在分析大型数据集时会变得非常耗时，因为它会在整个数据集中搜索相似数据点。此外，在高维数据集中，最近与最远邻居之间的差别非常小，因此KNN的准确性会降低。

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder

# 构造示例数据
df = pd.DataFrame({
    "用户ID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十", "钱十一", "陈十二"],
    "性别": ["男", "女", "男", "女", "男", "女", "男", "女", "男", "女"],
    "年龄": [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", "Beijing", "hangzhou", "Nanjing", "shenzhen", "Beijing"],
    "年收入": [80000, 120000, 75000, 95000, 120000, 110000, 98000, 105000, 88000, 92000],
    "上次消费金额": [200, 500, 300, np.nan, 500, 400, 250, 350, 280, 320]
})

print("原始数据：")
print(df)

# 准备KNN填充的数据
df_knn = df.copy()

# 对分类列进行编码
le = LabelEncoder()
df_knn["城市"] = df_knn["城市"].fillna("未知")
df_knn["城市"] = le.fit_transform(df_knn["城市"])
df_knn["性别"] = le.fit_transform(df_knn["性别"])

# 选择数值列进行KNN填充
numeric_cols = ["年龄", "年收入", "上次消费金额", "性别", "城市"]
df_numeric = df_knn[numeric_cols].copy()

print("\n编码后的数据（用于KNN填充）：")
print(df_numeric)

# KNN填充
knn_imputer = KNNImputer(n_neighbors=5)
df_knn_filled = pd.DataFrame(
    knn_imputer.fit_transform(df_numeric),
    columns=df_numeric.columns
)

# 将编码后的分类列还原
df_knn_filled["性别"] = le.inverse_transform(df_knn_filled["性别"].astype(int))
df_knn_filled["城市"] = le.inverse_transform(df_knn_filled["城市"].astype(int))

# 将填充后的值更新回原始DataFrame
df["年龄"] = df_knn_filled["年龄"]
df["上次消费金额"] = df_knn_filled["上次消费金额"]

print("\nKNN填充后的数据：")
print(df)

## 4. 回归填充（Regression）
---------------------------------------------------------------------

回归填充法是用已有数据建立回归模型，将缺失值作为因变量进行预测估计。

适用于缺失值与其他变量存在较强线性关系的情况。

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# 构造示例数据
df = pd.DataFrame({
    "用户ID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十", "钱十一", "陈十二"],
    "性别": ["男", "女", "男", "女", "男", "女", "男", "女", "男", "女"],
    "年龄": [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", "Beijing", "hangzhou", "Nanjing", "shenzhen", "Beijing"],
    "年收入": [80000, 120000, 75000, 95000, 120000, 110000, 98000, 105000, 88000, 92000],
    "上次消费金额": [200, 500, 300, np.nan, 500, 400, 250, 350, 280, 320]
})

print("原始数据：")
print(df)


def regression_impute(df, target_col, feature_cols):
    """
    回归填充法
    参数：
        df: DataFrame
        target_col: 需要填充的目标列
        feature_cols: 用于预测的特征列列表
    返回：
        填充后的DataFrame
    """
    df_reg = df.copy()
    
    # 分离有目标值和没有目标值的数据
    df_known = df_reg[df_reg[target_col].notna()].copy()
    df_unknown = df_reg[df_reg[target_col].isna()].copy()

    if len(df_unknown) == 0:
        print(f"{target_col} 列没有缺失值")
        return df_reg

    # 准备特征和目标
    X_train = df_known[feature_cols]
    y_train = df_known[target_col]
    
    # 建立回归模型
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # 预测缺失值
    X_predict = df_unknown[feature_cols]
    predicted_values = model.predict(X_predict)
    
    # 填充预测值
    df_reg.loc[df_reg[target_col].isna(), target_col] = predicted_values
    
    return df_reg


# 用年收入和上次消费金额预测年龄
df = regression_impute(df, "年龄", ["年收入", "上次消费金额"])

# 用年龄和年收入预测上次消费金额
df = regression_impute(df, "上次消费金额", ["年龄", "年收入"])

print("\n回归填充后的数据：")
print(df)

## 方法对比

| 方法 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| 条件平均值填充 | 考虑了分类变量影响 | 需要有足够的分组样本 | 分组明显的缺失值 |
| 热卡填充 | 保持数据分布 | 相似度定义主观 | 小规模、结构化数据 |
| KNN填充 | 多维考虑，抗噪声 | 计算开销大 | 中等规模数据 |
| 回归填充 | 利用变量关系 | 假设线性关系 | 变量间有较强线性关系 |